# DART — Publication Colab: random vs small LM vs 8B (4-bit) + self-repair

End-to-end **environment** rollouts (not static data). Produces:

- `logs/colab_experiment.json` — per-episode logs (`episode`, `reward`, `avg_reward`, `action_count`, `model`)
- `docs/figures/training_curve.png` — smoothed training returns (3 policies)
- `docs/figures/final_comparison_bars.png` — tail mean ± std
- `docs/figures/behavior_glucose.png` — one episode glucose: random vs trained small LM
- `docs/figures/self_repair_episodes.png` — council + self-repair with repair markers

**Efficiency:** large model uses **fewer updates**; edit `CONFIG` to scale up if you have credits/VRAM. **T4+** for 4-bit 8B.

## 1) Clone, install, repo root (edit `YOUR_REPO_URL` / `CLONE_DIR` if needed)


In [ ]:
import os, subprocess, sys
from pathlib import Path
from typing import Optional

YOUR_REPO_URL = "https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git"
CLONE_DIR = Path("/content/mano")
_marker = Path("scripts") / "train_reinforce_twin.py"

def _find_dart_root(base: Path) -> Optional[Path]:
    nested, root = base / "DART" / _marker, base / _marker
    if nested.is_file():
        return (base / "DART").resolve()
    if root.is_file():
        return base.resolve()
    return None

if not CLONE_DIR.is_dir():
    subprocess.run(["git", "clone", YOUR_REPO_URL, str(CLONE_DIR)], check=True)
REPO_ROOT = _find_dart_root(CLONE_DIR)
if REPO_ROOT is None and (CLONE_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--ff-only"], check=False)
    REPO_ROOT = _find_dart_root(CLONE_DIR)
if REPO_ROOT is None:
    raise FileNotFoundError(f"No {_marker} under {CLONE_DIR}")
os.chdir(REPO_ROOT)
os.environ["DART_REPO_ROOT"] = str(REPO_ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "-r", "requirements_hackathon.txt"], check=True)
print("REPO_ROOT:", REPO_ROOT)

## 2) GPU (Runtime → Change runtime type → T4 or better for 4-bit 8B)


## 3) Hugging Face (gated Llama) — set token; model id for 4-bit 8B


In [ ]:
import os
from huggingface_hub import login

_HF_PLACEHOLDER = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
os.environ["HF_TOKEN"] = _HF_PLACEHOLDER
os.environ["MODEL_ID"] = "unsloth/Meta-Llama-3.1-8B-Instruct-unsloth-bnb-4bit"
if os.environ["HF_TOKEN"].strip() and os.environ["HF_TOKEN"] != _HF_PLACEHOLDER:
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("MODEL_ID:", os.environ["MODEL_ID"])

## 4) CONFIG (tune for speed vs quality)
Increase `small.updates` / `large.updates` when you have time and budget.

In [ ]:
import os, torch, json
from pathlib import Path

REPO = Path(os.environ.get("DART_REPO_ROOT", ".")).resolve()
os.chdir(REPO)
Path("logs").mkdir(exist_ok=True)
Path("docs/figures").mkdir(parents=True, exist_ok=True)

CONFIG = {
    "max_steps": 20,
    "random_episodes": 40,
    "small": {
        "model_id": "distilgpt2",
        "short_label": "distilgpt2",
        "updates": 24,
        "episodes_per_update": 2,
        "max_new_tokens": 48,
    },
    "large": {
        "model_id": os.environ.get("MODEL_ID", "unsloth/Meta-Llama-3.1-8B-Instruct-unsloth-bnb-4bit"),
        "short_label": "llama-8b-4bit",
        "updates": 8,
        "episodes_per_update": 1,
        "load_4bit": True,
        "trust_remote_code": True,
        "max_new_tokens": 40,
    },
    "council_episodes": 16,
    "train_seed_base": 101,
    "ma_window": 5,
    "bar_tail_episodes": 15,
    "out_json": "logs/colab_experiment.json",
}
print(CONFIG)

## 5) Run baselines, train two LMs, council self-repair, save JSON + figures


In [ ]:
import sys, json, subprocess, os, torch
from pathlib import Path
REPO = Path(os.environ["DART_REPO_ROOT"]).resolve()
os.chdir(REPO)
sys.path.insert(0, str(REPO))

from training.colab_episode_rl import (
    collect_random_baseline,
    train_reinforce_with_episode_log,
    collect_trained_episode_glucose,
    council_self_repair_episode_log,
)

M = CONFIG["max_steps"]
sb = int(CONFIG["train_seed_base"])

r_rows, _, g_random = collect_random_baseline(
    n_episodes=int(CONFIG["random_episodes"]),
    max_steps=M,
    seed=0,
    model_key="random",
)

S = CONFIG["small"]
s_rows, model_s, tok_s, dev_s = train_reinforce_with_episode_log(
    model_id=S["model_id"],
    short_label=S["short_label"],
    load_in_4bit=False,
    trust_remote_code=False,
    updates=int(S["updates"]),
    episodes_per_update=int(S["episodes_per_update"]),
    max_steps=M,
    train_seed_base=sb,
    max_new_tokens=int(S.get("max_new_tokens", 48)),
)
g_trained = collect_trained_episode_glucose(
    model_s, tok_s, dev_s, max_steps=M, max_new_tokens=int(S.get("max_new_tokens", 48)), env_seed=10_100
)
del model_s, tok_s
torch.cuda.empty_cache() if torch.cuda.is_available() else None

L = CONFIG["large"]
l_rows = []
if torch.cuda.is_available():
    l_rows, model_l, tok_l, dev_l = train_reinforce_with_episode_log(
        model_id=L["model_id"],
        short_label=L["short_label"],
        load_in_4bit=bool(L.get("load_4bit", True)),
        trust_remote_code=bool(L.get("trust_remote_code", True)),
        updates=int(L["updates"]),
        episodes_per_update=int(L["episodes_per_update"]),
        max_steps=M,
        train_seed_base=sb + 5_000,
        max_new_tokens=int(L.get("max_new_tokens", 40)),
    )
    del model_l, tok_l
    torch.cuda.empty_cache()
else:
    print("[skip 8B] no CUDA; only random + distil in JSON.")

c_rows, repair_marks = council_self_repair_episode_log(
    n_episodes=int(CONFIG["council_episodes"]),
    max_steps=M,
    seed=7,
)

all_rows = r_rows + s_rows + l_rows + c_rows
out_path = REPO / CONFIG["out_json"]
payload = {
    "config": {k: v for k, v in CONFIG.items() if k != "out_json"},
    "episodes": all_rows,
    "glucose": {"random": g_random, "trained": g_trained},
    "self_repair_episodes": repair_marks,
    "plot_order": ["random", "distilgpt2", "llama-8b-4bit"],
    "bar_models": ["random", "distilgpt2", "llama-8b-4bit"],
    "bar_tail_episodes": int(CONFIG.get("bar_tail_episodes", 15)),
    "ma_window": int(CONFIG.get("ma_window", 5)),
}
out_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("wrote", out_path)

subprocess.run(
    [
        sys.executable, "scripts/plot_colab_publication.py", "--in-json", str(out_path),
        "--out-dir", "docs/figures", "--ma-window", str(int(CONFIG["ma_window"])),
    ],
    cwd=REPO, check=True,
)
print("plots in docs/figures/")

## 6) View figures in Colab (then commit to Git)

```bash
git add logs/colab_experiment.json docs/figures/training_curve.png \
  docs/figures/final_comparison_bars.png docs/figures/behavior_glucose.png \
  docs/figures/self_repair_episodes.png
git commit -m "Colab multi-model + publication figures"
```

**Note:** “trained > random” and “8B > distil” require enough updates and compute; if curves overlap, increase `CONFIG['small']['updates']` and `CONFIG['large']['updates']` (and HF time budget).

In [ ]:
from IPython.display import display, Image
from pathlib import Path
import os
F = Path(os.environ['DART_REPO_ROOT']) / 'docs' / 'figures'
for name in (
    'training_curve.png', 'final_comparison_bars.png',
    'behavior_glucose.png', 'self_repair_episodes.png',
):
    p = F / name
    if p.is_file():
        display(Image(filename=str(p)))